# TERRA graphs

Graphs for analysis and the report. The calculations run in TypeScript (`lib/science/`); this notebook only reads CSVs.

1. Refresh the data: `npm run export:graphs` (real Lipa CSVs) and `npx tsx scripts/export-synthetic-csv.ts` (synthetic record).
2. Run all cells.

Every title says what the data is: **REAL** (NASA pipeline), **SYNTHETIC** (simulated, planted problems) or **MODEL** (model output).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
lipa = pd.read_csv(ROOT / "notebooks/data/lipa_daily.csv", parse_dates=["date"])
water = pd.read_csv(ROOT / "notebooks/data/lipa_water.csv", parse_dates=["date"])
syn = pd.read_csv(ROOT / "experiments/samples/synthetic-field-seed1-fao56.csv", parse_dates=["date"])
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})

## 1. REAL — Lipa rain: 14-day IMERG total vs its own normal

The rain-departure rule compares the 14-day total with the normal for the same days (D-006). It alarms below 0.45× or above 1.7× the normal (shaded band).
Dots mark the engine's alarm onsets. POWER is shown for comparison only (F-010: the two sources disagree day to day).

In [ ]:
d = lipa.set_index("date")
imerg14 = d["rain_imerg_mm"].rolling(14, min_periods=14).sum()
power14 = d["rain_power_mm"].rolling(14, min_periods=14).sum()
normal14 = d["imerg_normal_mm_day"].rolling(14, min_periods=14).sum()

fig, ax = plt.subplots()
ax.fill_between(normal14.index, 0.45 * normal14, 1.7 * normal14, color="tab:blue", alpha=0.12, label="no-alarm band (0.45–1.7 × IMERG normal)")
ax.plot(normal14, color="tab:blue", lw=1, ls="--", label="IMERG normal, 14 days")
ax.plot(imerg14, color="tab:blue", lw=1.5, label="IMERG Final, 14-day total")
ax.plot(power14, color="tab:orange", lw=1, alpha=0.7, label="NASA POWER, 14-day total")
alarms = d[d["alarm"] == "rain-departure"]
ax.scatter(alarms.index, imerg14.reindex(alarms.index), color="black", zorder=3, s=18, label="alarm onset")
ax.set(title="REAL data — Lipa Cropland rain, 14-day totals (2024–2025)", ylabel="mm per 14 days")
ax.legend(loc="upper left", fontsize=8)
plt.show()

## 2. REAL — Lipa topsoil moisture from SMAP L3

One value per overpass day for a ~9 km cell. Every value is flagged "retrieval not recommended" and three different cells were used (F-009), so read the pattern, not the level.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(d.index, d["smap_m3m3"], s=8, color="tab:green")
ax.set(title="REAL data — SMAP L3 topsoil moisture over Lipa (all flagged, F-009)", ylabel="m³/m³")
plt.show()

## 3. SYNTHETIC — planted problems and the engine's alarms

A simulated record (seed 1, FAO-56 soil, station gaps). Shaded spans are planted problems; vertical lines are alarms the engine raised. Demo data: it tests the engine, not the field.

In [ ]:
s = syn.set_index("date")
fig, ax = plt.subplots()
ax.plot(s["soil_moisture_10cm_m3m3"], color="tab:green", label="station 10 cm")
ax.plot(s["soil_moisture_40cm_m3m3"], color="tab:brown", label="station 40 cm")
for kind, colour in {"dry-spell": "tab:red", "wet-spell": "tab:blue", "ndvi-dip": "tab:purple"}.items():
    days = s.index[s["planted_problem"].fillna("").str.contains(kind)]
    if len(days):
        ax.axvspan(days.min(), days.max() + pd.Timedelta(days=1), color=colour, alpha=0.15, label=f"planted: {kind}")
for day, kinds in s["alarm_raised"].dropna().items():
    ax.axvline(day, color="black", lw=0.6, alpha=0.5)
ax.set(title="SYNTHETIC data — soil moisture, planted problems (shaded) and alarms (lines)", ylabel="m³/m³")
ax.legend(loc="lower left", fontsize=8)
plt.show()

## 4. MODEL — crop water balance on the real Lipa weather

Model output (`lib/science/water/cropWater.ts`) driven by real IMERG rain (POWER on days IMERG is missing) and POWER ET0.
The crop and planting date (corn, 2024-06-01) are **assumed** and the soil is indicative, so this shows how the model behaves, not the field's state.
Water stress starts when depletion passes RAW.

In [ ]:
w = water.set_index("date")
fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
a1.plot(w["depletion_mm"], color="tab:red", label="root-zone depletion")
a1.plot(w["raw_mm"], color="black", ls="--", lw=1, label="RAW (stress starts)")
a1.plot(w["taw_mm"], color="grey", ls=":", lw=1, label="TAW (wilting point)")
a1.invert_yaxis(); a1.set(ylabel="mm below field capacity"); a1.legend(fontsize=8)
a2.plot(w["etc_mm"], color="tab:blue", lw=1, label="ETc = Kc × ET0 (demand)")
a2.plot(w["eta_mm"], color="tab:green", lw=1, label="ETa (actual use)")
a2.set(ylabel="mm/day"); a2.legend(fontsize=8)
fig.suptitle("MODEL output on REAL weather — Lipa, assumed corn planted 2024-06-01")
plt.show()